<a href="https://colab.research.google.com/github/jabri62018/Zx_Mother_Function_Jabri/blob/Zx_Mother_Function_Jabri/Zx_28.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:

# ZX_v1.3.6_zero-input with live progress
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import mpmath as mp
import pandas as pd
import time, sys
from matplotlib.backends.backend_pdf import PdfPages

mp.mp.dps = 35
plt.ioff()

# 1. تعريف Z(x) - دالة زيتا ريمان الحقيقية
def Z(x):
    x = mp.mpf(x)
    s = 0.5 + 1j*x
    log_zeta_prime = mp.diff(lambda z: mp.log(mp.zeta(z)), s)
    return 0.5 * x**2 * mp.re(log_zeta_prime)

def Zp(x): return mp.diff(Z, x)
def Zpp(x): return mp.diff(Zp, x)

# 2. حساب الجذور الـ28 مع شريط تقدم حي
print("=== Starting ZX Zero-Input Calculation ===", flush=True)
print("Finding 28 roots... هذا بياخذ 15-25 دقيقة، انتظر ⏳\n", flush=True)

roots = []
guesses = np.linspace(14, 300, 3000)
total_guesses = len(guesses)
start_time = time.time()
last_update = time.time()

def print_progress(i, found, elapsed):
    percent = (i + 1) / total_guesses * 100
    eta = elapsed / (i + 1) * (total_guesses - i - 1) if i > 0 else 0
    bar_len = 25
    filled = int(bar_len * (i + 1) / total_guesses)
    bar = '█' * filled + '░' * (bar_len - filled)
    sys.stdout.write(
        f"\r[{bar}] {percent:5.1f}% | Found: {found}/28 | "
        f"Elapsed: {int(elapsed//60):02d}:{int(elapsed%60):02d} | "
        f"ETA: {int(eta//60):02d}:{int(eta%60):02d} "
    )
    sys.stdout.flush()

for i, guess in enumerate(guesses):
    if len(roots) == 28:
        break

    # لا تبحث في كل نقطة عشان السرعة
    if i % 5!= 0 and len(roots) > 0:
        continue

    try:
        r = mp.findroot(Z, guess, solver='secant', maxsteps=30, tol=1e-20)
        r = float(r)
        if r > 10 and all(abs(r - rr) > 1e-6 for rr in roots):
            roots.append(r)
            # اطبع الجذر فوراً
            print(f"\n✓ Root {len(roots):2d}/28: γ = {r:.12f}", flush=True)
    except:
        pass

    # حدث شريط التقدم كل 0.5 ثانية
    now = time.time()
    if now - last_update > 0.5:
        print_progress(i, len(roots), now - start_time)
        last_update = now

print(f"\n\n✓ Done finding roots. Total found: {len(roots)} in {(time.time() - start_time)/60:.1f} min", flush=True)

if len(roots) < 6:
    raise RuntimeError(f"وجدت {len(roots)} جذور فقط. لازم 6 على الأقل عشان المقارنة")

# 3. حساب الثوابت مع مؤشر تقدم
print("\nCalculating C_n constants...", flush=True)
constants = []
start_calc = time.time()

for i, r in enumerate(roots):
    c = float(Zpp(r) / Zp(r))
    constants.append(c)
    # اطبع كل 5 جذور
    if (i+1) % 5 == 0 or i == len(roots)-1:
        elapsed = time.time() - start_calc
        print(f"Calculated C_{i+1}/{len(roots)} | Time: {elapsed:.1f}s", flush=True)

# 4. تطبيع ومقارنة
print("\nScaling and comparing with physics constants...", flush=True)
const_names = ["c","G","hbar","h","e","me","mp","alpha","mu0","eps0",
               "kB","NA","R","sigma","Ry","a0","muB","muN",
               "GF","m_nu","Lambda_QCD","H0","rho_c","TCMB","g","mp_MP","Lambda","tP"]

phys_vals = np.array([299792458.0, 6.674e-11, 1.054571817e-34, 6.62607015e-34, 1.602176634e-19,
                      9.1093837015e-31, 1.67262192369e-27, 7.2973525693e-3, 1.25663706212e-6,
                      8.8541878128e-12, 1.380649e-23, 6.02214076e23, 8.314462618, 5.670374419e-8,
                      10973731.568160, 5.29177210903e-11, 9.2740100783e-24, 5.0507837461e-27,
                      1.1663787e-5, 1e-36, 200e6, 2.2e-18, 8.5e-27, 2.725, 9.80665, 4.593e-20,
                      1.1e-52, 5.391247e-44])

C_abs = np.abs(constants[:6])
S = phys_vals[1] / C_abs[1]
C_scaled = C_abs * S

# عرض النتائج كاملة على الشاشة أول
print("\n" + "="*60, flush=True)
print("=== Zero-Input Results ===", flush=True)
print("="*60, flush=True)
matched_count = 0
for i in range(6):
    err = abs(C_scaled[i] - phys_vals[i]) / phys_vals[i]
    status = "Matched" if err < 0.15 else "Failed"
    if status == "Matched":
        matched_count += 1
    print(f"C{i+1} {const_names[i]:<6}: ZX={C_scaled[i]:.6e} Phys={phys_vals[i]:.6e} Err={err:.1%} [{status}]", flush=True)
print("-"*60, flush=True)
print(f"Result: {matched_count}/6 constants matched within 15%", flush=True)
print("="*60, flush=True)
print("\n[INFO] All results displayed above. Now saving files silently...", flush=True)

# 5. حفظ CSV بصمت
df_roots = pd.DataFrame({"n": range(1,len(roots)+1), "gamma_n": roots, "C_n": constants})
df_roots.to_csv("ZX_roots.csv", index=False)

df_const = pd.DataFrame({
    "n": range(1,7),
    "Constant": const_names[:6],
    "ZX_Scaled": C_scaled,
    "Physics": phys_vals[:6],
    "Rel_Error": [abs(C_scaled[i]-phys_vals[i])/phys_vals[i] for i in range(6)]
})
df_const.to_csv("ZX_constants.csv", index=False)

# 6. رسم وحفظ PNG بصمت
fig1 = plt.figure(figsize=(10,6))
plt.semilogy(roots[:10], np.abs(constants[:10]), 'ro-')
plt.xlabel("n"); plt.ylabel("|C_n|"); plt.title("ZX Constants - Zero Input")
plt.grid(True, alpha=0.3)
plt.savefig("ZX_plot.png", dpi=300, bbox_inches='tight')
plt.close(fig1)

fig2 = plt.figure(figsize=(10,6))
plt.loglog(range(1,7), C_scaled, 'bo-', label='ZX')
plt.loglog(range(1,7), phys_vals[:6], 'ro-', label='Physics')
plt.xlabel("n"); plt.ylabel("Scaled Value"); plt.legend()
plt.title("ZX vs Physics - After Scaling")
plt.grid(True, alpha=0.3)
plt.savefig("ZX_compare.png", dpi=300, bbox_inches='tight')
plt.close(fig2)

# 7. تجميع PDF بصمت
with PdfPages("ZX_report.pdf") as pdf:
    fig = plt.figure(figsize=(8.27,11.69))
    fig.text(0.5, 0.8, "ZX Model - Zero Input Report", ha='center', fontsize=20, weight='bold')
    fig.text(0.5, 0.7, f"Roots Found: {len(roots)}", ha='center', fontsize=14)
    fig.text(0.5, 0.6, f"Matched Constants: {matched_count}/6", ha='center', fontsize=14)
    fig.text(0.5, 0.5, "Sana'a, Yemen", ha='center', fontsize=12)
    pdf.savefig(fig); plt.close(fig)
    for img in ["ZX_plot.png", "ZX_compare.png"]:
        fig = plt.figure(figsize=(8.27,11.69))
        plt.imshow(plt.imread(img)); plt.axis('off')
        pdf.savefig(fig); plt.close(fig)

print("\n=== DONE ===", flush=True)
print("Files saved: ZX_roots.csv, ZX_constants.csv, ZX_plot.png, ZX_compare.png, ZX_report.pdf", flush=True)

=== Starting ZX Zero-Input Calculation ===
Finding 28 roots... هذا بياخذ 15-25 دقيقة، انتظر ⏳

[█████░░░░░░░░░░░░░░░░░░░░]  23.0% | Found: 0/28 | Elapsed: 01:31 | ETA: 05:05 